# Collapsing and Scaling Tensors

```
Copyright 2025 National Technology & Engineering Solutions of Sandia,
LLC (NTESS). Under the terms of Contract DE-NA0003525 with NTESS, the
U.S. Government retains certain rights in this software.
```

This tutorial shows how to reduce a tensor along some of its dimensions with `collapse` (for example, summing or taking maxima) and how to scale slices of a tensor with `scale`. Both are available for dense `tensor`s and sparse `sptensor`s.

[API reference](../utilities.rst), [tensor](../tensor.rst), [sptensor](../sptensor.rst)

In [ ]:
from __future__ import annotations

import numpy as np

import pyttb as ttb

## Overview
* `collapse(dims, fun)` applies a function (the sum, by default) over the specified dimensions and returns the result with those dimensions removed.
* `scale(factor, dims)` multiplies each slice along the specified dimensions by a factor.

### Setup

In [ ]:
def shape_of(t):
    # Show shapes as plain python integers
    return tuple(int(n) for n in t.shape)


T = ttb.tensor(np.arange(1, 25), shape=(3, 4, 2))
S = T.to_sptensor()  # the same data as a sparse tensor
print(shape_of(T))

## Collapsing a tensor (`collapse`)
### Summing
With no arguments, `collapse` sums all of the entries and returns a scalar. With `dims`, only those dimensions are summed and the remaining dimensions form the result.

In [ ]:
print(T.collapse())
print(shape_of(T.collapse(0)))  # sum over mode 0
print(shape_of(T.collapse([0, 2])))  # sum over modes 0 and 2
print(S.collapse())
print(shape_of(S.collapse(0)))

### Other functions
Any function that reduces a one-dimensional array to a scalar can be used. Here we compute the maximum over mode 0 and the mean over modes 0 and 1.

In [ ]:
print(T.collapse(0, np.max).data[:, 0])
print(T.collapse([0, 1], np.mean))
print(shape_of(S.collapse(0, np.max)))

### Collapse is the building block for sums over slices
For example, the sum of squares of each mode-2 slice gives the squared Frobenius norm of each slice.

In [ ]:
slice_norms_squared = (T**2).collapse([0, 1])
print(slice_norms_squared)
print(np.isclose(slice_norms_squared.data.sum(), T.norm() ** 2))

## Scaling a tensor (`scale`)
### Scaling along one dimension
The factor is an array with one entry for each index of the scaled dimension. Here we scale the two mode-2 slices by 10 and 100.

In [ ]:
factor = np.array([10.0, 100.0])
T_scaled = T.scale(factor, 2)
print(T.data[0, 0, :])
print(T_scaled.data[0, 0, :])
S_scaled = S.scale(factor, 2)
print(S_scaled.full().isequal(T_scaled))

### Scaling by a tensor
The factor can also be a `tensor`, which is useful for scaling along several dimensions at once. Here the factor has one entry for each (mode 0, mode 1) slice.

In [ ]:
factor_tensor = ttb.tensor(np.arange(1, 13), shape=(3, 4))
T_scaled2 = T.scale(factor_tensor, [0, 1])
print(T_scaled2.data[:, :, 0] / T.data[:, :, 0])

### Normalizing slices
Combining the two operations normalizes slices: divide each mode-2 slice by its norm.

In [ ]:
norms = np.sqrt((T**2).collapse([0, 1]).data)
T_normalized = T.scale(1 / norms, 2)
print((T_normalized**2).collapse([0, 1]))

## Summary
`collapse` and `scale` are methods of the [tensor](../tensor.rst) and [sptensor](../sptensor.rst) classes; see their API pages for details.